In [1]:
!pip install langchain openai langsmith python-dotenv


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

False

In [3]:
from langchain.prompts import PromptTemplate

extract_template = """
You are an AI resume parser.

Extract the following from the resume:
- Skills
- Experience
- Tools

Rules:
- Do NOT assume anything
- Only extract what is explicitly mentioned

Resume:
{resume}

Output format (STRICT JSON):
{{
 "skills": [],
 "experience": "",
 "tools": []
}}
"""

extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template=extract_template
)

ModuleNotFoundError: No module named 'langchain.prompts'

In [4]:
match_template = """
Compare the candidate profile with job description.

Job Description:
{jd}

Candidate Data:
{extracted_data}

Return:
- Matching Skills
- Missing Skills
"""

match_prompt = PromptTemplate(
    input_variables=["jd", "extracted_data"],
    template=match_template
)

NameError: name 'PromptTemplate' is not defined

In [5]:
score_template = """
Based on the match result, assign a score from 0 to 100.

Rules:
- Strong match → 80-100
- Moderate → 50-79
- Weak → below 50

Match Data:
{match_data}

Return ONLY a number.
"""

score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template=score_template
)

NameError: name 'PromptTemplate' is not defined

In [6]:
explain_template = """
Explain why this candidate received the score.

Include:
- Strengths
- Weaknesses
- Final justification

Score: {score}
Match Data: {match_data}
"""

explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template=explain_template
)

NameError: name 'PromptTemplate' is not defined

In [7]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

ImportError: cannot import name 'ChatOpenAI' from 'langchain.chat_models' (C:\Users\Hamsini\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain\chat_models\__init__.py)

In [ ]:
# Extraction Chain
extract_chain = extract_prompt | llm

# Matching Chain
match_chain = match_prompt | llm

# Scoring Chain
score_chain = score_prompt | llm

# Explanation Chain
explain_chain = explain_prompt | llm

In [ ]:
job_description = """
Looking for Data Scientist with:
- Python
- Machine Learning
- SQL
- Deep Learning
"""

strong_resume = """
3 years experience in Data Science.
Skills: Python, Machine Learning, Deep Learning, SQL
Tools: TensorFlow, Pandas
"""

average_resume = """
1 year experience.
Skills: Python, SQL
Tools: Excel
"""

weak_resume = """
Fresher.
Skills: MS Office
"""

In [ ]:
def run_pipeline(resume, jd):
    
    # Step 1: Extract
    extracted = extract_chain.invoke({"resume": resume})
    
    # Step 2: Match
    match = match_chain.invoke({
        "jd": jd,
        "extracted_data": extracted.content
    })
    
    # Step 3: Score
    score = score_chain.invoke({
        "match_data": match.content
    })
    
    # Step 4: Explain
    explanation = explain_chain.invoke({
        "score": score.content,
        "match_data": match.content
    })
    
    return {
        "extracted": extracted.content,
        "match": match.content,
        "score": score.content,
        "explanation": explanation.content
    }

In [ ]:
print("STRONG CANDIDATE\n", run_pipeline(strong_resume, job_description))
print("\nAVERAGE CANDIDATE\n", run_pipeline(average_resume, job_description))
print("\nWEAK CANDIDATE\n", run_pipeline(weak_resume, job_description))

In [ ]:
bad_resume = "I know everything in AI"

print(run_pipeline(bad_resume, job_description))

In [8]:
import json

def safe_json_parse(text):
    try:
        return json.loads(text)
    except:
        return text

In [ ]:
Resume
   ↓
Skill Extraction
   ↓
Matching
   ↓
Scoring
   ↓
Explanation
   ↓
LangSmith Trace